# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR² colorectal cancer dataset using the `mlcroissant` library.

### Dataset Source
The dataset is specified by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

This dataset details clinical, demographic, pathological, and molecular characteristics of 77 cancer survivors with second primary colorectal cancer.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
We load the dataset metadata and tabular records using `mlcroissant`. The Croissant schema describes the complete structure, including record sets, fields, and associated files.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata (this does not load the data yet)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset loaded: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
We review all available record sets and their fields/columns, referencing each by its Croissant `@id` for reproducibility.

Let's display all record sets, their IDs, and their field details.

In [ ]:
# List all record sets associated with the dataset
record_sets = list(dataset.record_sets.values())
print(f"Found {len(record_sets)} record set(s):")

for rs in record_sets:
    print(f"\nRecord Set: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Number of fields: {len(rs.fields)}")
    for field in rs.fields:
        print(f"    - Field: {field.name}")
        print(f"      @id: {field.id}")
        print(f"      Data type: {field.data_type}")

## 3. Data Extraction
We demonstrate how to load the records from a specific record set into a `pandas` DataFrame, using Croissant `@id`s throughout.

Below, we load all record sets by their `@id` and show columns for one example DataFrame.

In [ ]:
# Prepare a list of record set @ids
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set '@id': {record_set_id}")

# Show columns for the first record set for demonstration
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"\nFields (@id as column names) in record set '@id': {example_record_set_id}")
    print(list(dataframes[example_record_set_id].columns))
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
In this section, we show data processing using field and record set `@id`s. We'll filter on one numerical field, normalize values, and group by a categorical field.

First, let's enumerate all numeric fields available in our main record set.

In [ ]:
# Identify main record set (typically the largest or most relevant table)
main_rs = max(record_sets, key=lambda rs: len(rs.fields))  # Heuristic: pick one with the most fields
main_rs_id = main_rs.id

# Find numeric fields (int or float) by @id
numeric_fields = [field for field in main_rs.fields if field.data_type in ("Number", "Integer", "Float")]

if not numeric_fields:
    print("No numeric fields found in the main record set.")
else:
    print("Numeric fields available (with @id):")
    for field in numeric_fields:
        print(f"- {field.name} (@id: {field.id}) [{field.data_type}]")

In [ ]:
# Choose a numeric field by @id (replace with one found above if desired)
if numeric_fields:
    numeric_field = numeric_fields[0].id
    print(f"Using numeric field: {numeric_field}")

    # Display value distribution
    main_df = dataframes[main_rs_id]

    # Ensure field exists and is numeric
    if numeric_field in main_df.columns:
        # Convert to numeric (in case)
        main_df[numeric_field] = pd.to_numeric(main_df[numeric_field], errors='coerce')

        # Filter values above the mean (as an example threshold)
        threshold = main_df[numeric_field].mean()
        filtered_df = main_df[main_df[numeric_field] > threshold].copy()
        print(f"Filtered records where {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} records.")

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Choose a grouping field (categorical) by @id
        categorical_fields = [f for f in main_rs.fields if f.data_type in ("Text", "Boolean")]
        if categorical_fields:
            group_field = categorical_fields[0].id
            if group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
                print(f"Grouped mean of {numeric_field} by {group_field}:")
                print(grouped_df)
    else:
        print(f"Field {numeric_field} not found in DataFrame columns.")
else:
    print("No numeric fields available -- skipping EDA example.")

## 5. Visualization
Let's visualize the distribution of the numeric field and group means. We use `matplotlib` and `seaborn` for standard plots, but only for available fields as above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    field_name = numeric_fields[0].id
    df = dataframes[main_rs_id]
    if field_name in df.columns:
        df[field_name] = pd.to_numeric(df[field_name], errors='coerce')

        plt.figure(figsize=(8, 5))
        sns.histplot(df[field_name].dropna(), bins=12, kde=True)
        plt.title(f"Distribution of {field_name} in record set '@id': {main_rs_id}")
        plt.xlabel(field_name)
        plt.ylabel("Count")
        plt.show()

        # Barplot for group means if grouping was possible
        categorical_fields = [f for f in main_rs.fields if f.data_type in ("Text", "Boolean")]
        if categorical_fields:
            group_field = categorical_fields[0].id
            if group_field in df.columns:
                group_means = df.groupby(group_field)[field_name].mean().dropna()
                group_means.plot(kind='bar', figsize=(8,4))
                plt.title(f"Average {field_name} grouped by {group_field}")
                plt.ylabel(f"Mean {field_name}")
                plt.xlabel(group_field)
                plt.tight_layout()
                plt.show()
else:
    print("No numeric fields available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated step-by-step loading, exploration, and basic EDA of a clinical colorectal cancer survivor dataset via its Croissant schema using `mlcroissant`. All entities were referenced by their Croissant `@id` to maximize interoperability and reproducibility.

Key points:
- **Dataset loaded from schema**: All metadata and records are described and accessible via the Croissant schema.
- **`@id` referencing**: Record sets and fields are referenced and manipulated exclusively by `@id`.
- **Extensible EDA**: You can extend with custom filters, aggregations, and domain-specific analyses. Consult the Croissant metadata to discover available fields and their IDs.

For more advanced analyses, further consult the dataset's field descriptions and utilize additional `mlcroissant` capabilities to join, join-on, or extract features across record sets.

For issues with dataset access or schema, see the [FAIR² documentation](https://sen.science/) or open an issue with the repository.